# Лабораторна робота № 8
## Методи кластеризації: K-Means, метод ліктя, коефіцієнт силуета та PCA

**Мета:** навчитися застосовувати алгоритм K-Means, добирати кількість кластерів за допомогою методу ліктя та коефіцієнта силуета; дослідити використання PCA. Додатково — порівняти моделі класифікації на наборі даних про студентів (цей етап збережено з початкової роботи).

**Середовище:** Google Colab або Jupyter Notebook із Python. Для завдання 2 потрібно завантажити набір даних з кагл https://www.kaggle.com/datasets/rabieelkharoua/students-performance-dataset?select=Student_performance_data+_.csv.

**Виконання:** запускайте комірки послідовно, зверху вниз.

## Завдання 1. Кластеризація набору даних Iris

**1.1. Імпорт бібліотек.** Імпортуйте `matplotlib`, `pandas`, `seaborn`, `load_iris`, `KMeans`, `silhouette_score` і `StandardScaler`.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

**1.2. Завантаження Iris.** Завантажте Iris та виведіть перші п’ять рядків таблиці з назвами ознак.

In [ ]:
iris = load_iris()
iris_df = pd.DataFrame(iris.data, columns=iris.feature_names)
display(iris_df.head())

**1.3. Масштабування та метод ліктя.** Стандартизуйте дані та за допомогою методу ліктя визначте оптимальну кількість кластерів

In [ ]:
iris_features = iris_df.iloc[:, :3]
iris_scaled = StandardScaler().fit_transform(iris_features)
ks = range(1, 11)
iris_wcss = [KMeans(n_clusters=k, random_state=42, n_init=10).fit(iris_scaled).inertia_ for k in ks]
plt.plot(list(ks), iris_wcss, marker='o')
plt.title('Iris: метод ліктя')
plt.xlabel('Кількість кластерів'); plt.ylabel('WCSS'); plt.grid(); plt.show()

**1.4. Метод силуетів.** Для тих самих трьох ознак Iris обчисліть середній коефіцієнт силуета для 2–10 кластерів і побудуйте графік.

In [ ]:
iris_sil = [silhouette_score(iris_scaled, KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(iris_scaled)) for k in range(2, 11)]
plt.plot(range(2, 11), iris_sil, marker='o')
plt.title('Iris: коефіцієнт силуета')
plt.xlabel('Кількість кластерів'); plt.ylabel('Silhouette'); plt.grid(); plt.show()
print('Найкраще k за силуетом:', range(2, 11)[iris_sil.index(max(iris_sil))])

**1.5. Фінальна кластеризація та візуалізація.** Побудуйте K-Means для `n_clusters=2`  На площині покажіть перші дві з них та відповідні координати центрів. Це **двовимірне відображення результату кластеризації за трьома ознаками**.

In [ ]:
iris_model = KMeans(n_clusters=2, random_state=42, n_init=10)
iris_labels = iris_model.fit_predict(iris_scaled)
plt.scatter(iris_scaled[:, 0], iris_scaled[:, 1], c=iris_labels, cmap='viridis', alpha=0.7)
plt.scatter(iris_model.cluster_centers_[:, 0], iris_model.cluster_centers_[:, 1], c='red', marker='X', s=180, label='Центри')
plt.title('Iris: 2 кластери (перші дві з трьох ознак)')
plt.xlabel(iris_features.columns[0]); plt.ylabel(iris_features.columns[1]); plt.legend(); plt.show()

**Короткий висновок до завдання 1.** Метод ліктя показує, як зменшується WCSS зі збільшенням кількості кластерів. За методом силуетів найкраще значення визначається максимумом графіка. У завданні використано **2 кластери за умовою викладача**, навіть якщо метод ліктя або силуетів пропонує іншу кількість. Візуалізація показує перші дві з трьох стандартизованих ознак.

## Завдання 2. Дослідження даних студентів

Спочатку порівняйте три класифікатори (як у початковій лабораторній роботі), а потім виконайте кластеризацію. **Класифікація та кластеризація — різні задачі:** у класифікації використовується цільова ознака `GradeClass`, а в кластеризації — ні.

**2.1. Імпорт бібліотек для класифікації.** Імпортуйте засоби для поділу даних, добору параметрів, класифікації та оцінювання моделей.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from sklearn.decomposition import PCA
import os

**2.2. Завантаження файлу.** Помістіть `Student_data.csv` до робочої папки або завантажте його через Colab. Файл містить дані з Вашої початкової роботи.

In [ ]:
# Якщо CSV уже є у Colab, просто читаємо його. Інакше завантажуємо вручну.
filename = 'Student_performance_data _.csv'
if not os.path.exists(filename):
    try:
        from google.colab import files
        print('Завантажте файл Student_performance_data _.csv')
        files.upload()
    except ImportError:
        # Для локального запуску поруч із блокнотом
        filename = '/mnt/data/Student_performance_data _.csv'
print('Файл:', filename)

**2.3. Перегляд даних.** Прочитайте CSV у DataFrame і перегляньте вміст.

In [ ]:
df = pd.read_csv(filename)
display(df.head())

**2.4. Назви стовпців.** Виведіть назви всіх стовпців.

In [ ]:
print(df.columns.tolist())

**2.5. Структура таблиці.** Перегляньте типи стовпців і кількість непорожніх значень.

In [ ]:
df.info()

**2.6. Пропущені значення.** Перевірте, чи містить набір даних пропущені значення.

In [ ]:
print(df.isnull().sum())

**2.7. Дублікати.** Обчисліть кількість дублікатів.

In [ ]:
print('Кількість дублікатів:', df.duplicated().sum())

**2.8. Описова статистика.** Виведіть основні описові статистики.

In [ ]:
display(df.describe())

**2.9. Підготовка даних для класифікації.** Сформуйте `X` та `y`, прибравши з ознак цільову змінну `GradeClass` і службовий ідентифікатор `StudentID`. Поділіть дані у співвідношенні 80/20 та стандартизуйте ознаки.

In [ ]:
X = df.drop(columns=['StudentID', 'GradeClass'])
y = df['GradeClass'].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print('Навчальна вибірка:', X_train.shape, 'Тестова:', X_test.shape)

### 2.10. Класифікація: Random Forest

**Модель Random Forest.** Здійсніть добір гіперпараметрів `GridSearchCV` (5 згорток), виведіть найкращі параметри, середню CV-точність і тестову accuracy. Пошук може тривати довше через велику сітку параметрів.

In [ ]:
rf_search = GridSearchCV(RandomForestClassifier(random_state=42), {'n_estimators': [50, 100], 'max_depth': [5, 10, None]}, cv=5, scoring='accuracy', n_jobs=-1)
rf_search.fit(X_train_scaled, y_train)
rf_acc = accuracy_score(y_test, rf_search.predict(X_test_scaled))
print('Параметри:', rf_search.best_params_)
print('CV accuracy:', round(rf_search.best_score_, 4), 'Test accuracy:', round(rf_acc, 4))

### 2.11. Класифікація: Logistic Regression

**Модель Logistic Regression.** Застосуйте `GridSearchCV` до логістичної регресії. Для багатокласової задачі використайте `saga` — `liblinear` у нових версіях sklearn не підтримує багатокласову класифікацію без додаткової обгортки.

In [ ]:
lr_search = GridSearchCV(LogisticRegression(solver='saga', max_iter=3000, random_state=42), {'C': [0.1, 1, 10]}, cv=5, scoring='accuracy', n_jobs=-1)
lr_search.fit(X_train_scaled, y_train)
lr_acc = accuracy_score(y_test, lr_search.predict(X_test_scaled))
print('Параметри:', lr_search.best_params_)
print('CV accuracy:', round(lr_search.best_score_, 4), 'Test accuracy:', round(lr_acc, 4))

### 2.12. Класифікація: Support Vector Machine

**Модель SVM.** Виконайте підбір параметрів SVM та обчисліть CV-точність і тестову accuracy.

In [ ]:
svm_search = GridSearchCV(SVC(), {'C': [0.1, 1, 10], 'kernel': ['linear', 'rbf']}, cv=5, scoring='accuracy', n_jobs=-1)
svm_search.fit(X_train_scaled, y_train)
svm_acc = accuracy_score(y_test, svm_search.predict(X_test_scaled))
print('Параметри:', svm_search.best_params_)
print('CV accuracy:', round(svm_search.best_score_, 4), 'Test accuracy:', round(svm_acc, 4))

### 2.13. Порівняння результатів

**Порівняння моделей.** Порівняйте тестову точність трьох моделей. Оберіть модель **за найкращою CV-точністю**, а звіт `classification_report` виведіть один раз на тестових даних.

In [ ]:
models = {'Random Forest': rf_search, 'Logistic Regression': lr_search, 'SVM': svm_search}
comparison = pd.DataFrame({'Модель': list(models), 'CV accuracy': [m.best_score_ for m in models.values()], 'Test accuracy': [accuracy_score(y_test, m.predict(X_test_scaled)) for m in models.values()]})
display(comparison.round(4))
best_name = comparison.loc[comparison['CV accuracy'].idxmax(), 'Модель']
best_model = models[best_name]
print('Найкраща модель за CV:', best_name)
print(classification_report(y_test, best_model.predict(X_test_scaled), zero_division=0))

### 2.14. Кластеризація даних студентів: метод ліктя

**Метод ліктя для студентів.** Для кластеризації використайте чотири числові ознаки `Age`, `StudyTimeWeekly`, `Absences`, `GPA`. Стандартизуйте їх і побудуйте графік WCSS для 1–10 кластерів.

In [ ]:
student_features = df[['Age', 'StudyTimeWeekly', 'Absences', 'GPA']]
student_scaled = StandardScaler().fit_transform(student_features)
student_wcss = [KMeans(n_clusters=k, random_state=42, n_init=10).fit(student_scaled).inertia_ for k in range(1, 11)]
plt.plot(range(1, 11), student_wcss, marker='o')
plt.title('Студенти: метод ліктя')
plt.xlabel('Кількість кластерів'); plt.ylabel('WCSS'); plt.grid(); plt.show()

### 2.15. Кластеризація даних студентів: метод силуетів

**Метод силуетів для студентів.** Для тих самих чотирьох ознак побудуйте графік коефіцієнта силуета для 2–10 кластерів.

In [ ]:
student_sil = [silhouette_score(student_scaled, KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(student_scaled), sample_size=1000, random_state=42) for k in range(2, 11)]
plt.plot(range(2, 11), student_sil, marker='o')
plt.title('Студенти: метод силуетів')
plt.xlabel('Кількість кластерів'); plt.ylabel('Silhouette'); plt.grid(); plt.show()
print('Найкраще k за силуетом:', range(2, 11)[student_sil.index(max(student_sil))])

### 2.16. Додатковий підхід: PCA та автоматичне визначення точки ліктя

Тут використовується **інший набір ознак**: усі стовпці, крім `StudentID` та `GradeClass`. Після PCA кластеризацію проведемо за вибраними головними компонентами.

**Установлення додаткових пакетів.** За потреби встановіть `kneed` і `yellowbrick` у середовищі ноутбука.

In [ ]:
# Додаткові бібліотеки встановлюються лише за потреби.
import importlib.util
if importlib.util.find_spec('kneed') is None or importlib.util.find_spec('yellowbrick') is None:
    %pip -q install kneed yellowbrick

**Імпорт PCA і KneeLocator.** Імпортуйте `PCA`, `KneeLocator` та `numpy`.

In [ ]:
from sklearn.decomposition import PCA
from kneed import KneeLocator
import numpy as np

**Підготовка всіх ознак.** Приберіть `StudentID` і `GradeClass`, залишивши ознаки для PCA.

In [ ]:
pca_features = df.drop(columns=['StudentID', 'GradeClass'])
print('Ознак для PCA:', pca_features.shape[1])

**Модель PCA.** Стандартизуйте ознаки, навчіть PCA та знайдіть частку поясненої дисперсії кожної компоненти.

In [ ]:
pca_scaled = StandardScaler().fit_transform(pca_features)
pca_full = PCA().fit(pca_scaled)
variance = pca_full.explained_variance_ratio_
print('Пояснена дисперсія компонент:', np.round(variance, 4))

**Визначення кількості компонент.** Застосуйте `KneeLocator` до часток поясненої дисперсії. У підписі графіка вказуйте номер компоненти без додавання одиниці.

In [ ]:
component_numbers = np.arange(1, len(variance) + 1)
pca_knee = KneeLocator(component_numbers, variance, curve='convex', direction='decreasing')
n_components = int(pca_knee.knee) if pca_knee.knee is not None else int(np.searchsorted(np.cumsum(variance), 0.8) + 1)
plt.plot(component_numbers, variance, marker='o')
plt.axvline(n_components, color='red', linestyle='--', label=f'{n_components} компонент')
plt.title('PCA: пояснена дисперсія кожної компоненти')
plt.xlabel('Номер компоненти'); plt.ylabel('Частка дисперсії'); plt.legend(); plt.grid(); plt.show()
print('Обрано компонент:', n_components)

**Кумулятивна пояснена дисперсія.** На горизонтальній осі має бути **кількість головних компонент** (1, 2, 3, ...), тому індекс масиву для `n_components` дорівнює `n_components - 1`.

**Кумулятивна дисперсія та перетворення PCA.** Побудуйте графік кумулятивної дисперсії та позначте знайдену кількість компонент. Виведіть фактичну частку дисперсії. Перетворіть ознаки через PCA для наступного етапу.

In [ ]:
cumulative = np.cumsum(variance)
plt.plot(component_numbers, cumulative, marker='o')
plt.axvline(n_components, color='red', linestyle='--', label=f'{n_components} компонент')
plt.title('PCA: кумулятивна пояснена дисперсія')
plt.xlabel('Кількість компонент'); plt.ylabel('Частка дисперсії'); plt.legend(); plt.grid(); plt.show()
print('Пояснена дисперсія:', round(cumulative[n_components - 1] * 100, 2), '%')
pca_data = PCA(n_components=n_components).fit_transform(pca_scaled)
print('Розмірність після PCA:', pca_data.shape)

Зауваження: кумулятивна пояснена дисперсія — це **частка дисперсії**, а не відсоток «інформативності» даних. Значення потрібно брати з розрахунку, а не задавати наперед (наприклад, 32%).

**Імпорт KElbowVisualizer.** Імпортуйте `KElbowVisualizer` із `yellowbrick.cluster`.

In [ ]:
from yellowbrick.cluster import KElbowVisualizer

**Метод ліктя після PCA.** Побудуйте графік `KElbowVisualizer` для K-Means. Для узгодженості з попереднім етапом використайте **дані після PCA**.

In [ ]:
visualizer = KElbowVisualizer(KMeans(random_state=42, n_init=10), k=(2, 11), timings=False)
visualizer.fit(pca_data)
visualizer.show()
print('Кількість кластерів за ліктем після PCA:', visualizer.elbow_value_)

## Висновки

Під час лабораторної роботи досліджено методи кластеризації K-Means, ліктя, силуетів та PCA. Для Iris побудовано графіки та виконано кластеризацію на два кластери відповідно до умови. Для студентів проведено класифікацію за `GradeClass` за допомогою Random Forest, Logistic Regression та SVM; найкраща модель обирається **за CV accuracy**. На відміну від класифікації, кластеризація групує студентів **без використання `GradeClass`**. Для чотирьох ознак студентів порівняно метод ліктя й коефіцієнт силуета, а після PCA повторно використано метод ліктя. Точні значення кількості кластерів, обраних компонент і поясненої дисперсії наведені в результатах нижче.


In [ ]:
print('ПІДСУМКИ ЛАБОРАТОРНОЇ РОБОТИ №8')
print('Iris: найкраще k за силуетом =', range(2, 11)[iris_sil.index(max(iris_sil))], '; у завданні використано k=2')
print('Студенти: найкраще k за силуетом =', range(2, 11)[student_sil.index(max(student_sil))])
print('Найкращий класифікатор за CV =', best_name)
print('PCA: кількість компонент =', n_components, '; пояснена дисперсія =', f'{cumulative[n_components-1]*100:.2f}%')
print('Метод ліктя після PCA: k =', visualizer.elbow_value_)